# FFM48k_Figures — 48 kHz 自编码器的论文图（FFM48k_meas，只用验证集）

**只评估、只画图，不训练，不改任何检查点。** 论文里 48 kHz 的重建结果以 FFM48k_meas 为准；FFM48k_measim 只读它的 FiLM 参数。

**规则（写 notebook 时定好，跑之前不改）**

- **只用验证集**：FFM48k_meas 的测试集还没用过。本 notebook 不创建测试分区，一个测试受试者都不读（Cell 3 有断言）。
- **复现关卡**：画图之前，用同一次遍历的 RMSE 复现 PROJECT_LOG / `ae_lsd_report.json` 里记录的验证集数字（容差 0.005 dB），不通过就停。
- **画图关卡**：每个被选中的受试者，画图用的重建要重新给出选人时记录的 LSD（差 < 0.001 dB），保证右耳镜像进出没有写错。
- **选哪个人**：每个库选两耳平均 LSD 处于该库验证集**中位数**的受试者（偶数人取偏小的那个）。不挑最好看的。
- **频率**：HEALPix 图用 0.5、4、8、12 kHz 最近的频点（图上标实际频率）。**方向**：和 Unit A 相同的 5 个方向。
- 所有图都标出仰角 −45°：三个库在它下面没有测量，是插值外推出来的（§2.3）。

**输出**（`HRTF_Checkpoints/ae48k_figures/`，每张图 PNG 300 dpi + PDF；最后打包成 zip 下载）

| 文件 | 内容 | 建议放在 |
|---|---|---|
| `ae48k_directions_<库>.png` | 中位数受试者，5 个方向，两耳实测（实线）和重建（虚线） | 4.1.2（正文放一个库，其余可放附录） |
| `ae48k_healpix_<库>.png` | 同一个人左耳，4 个频率 × 实测 / 重建 / 绝对误差 | 4.1.2 |
| `ae48k_error_by_frequency.png` | 各库验证集逐频点 RMSE（模型 / 均值基线）及其比值 | 4.1.4 Evaluation（notch 频段那段） |
| `ae48k_error_maps.png` | 各库验证集逐方向 RMSE（左耳坐标系） | 4.1.4 Evaluation（HUTUBS 那段） |
| `ae48k_film_meas.png`、`ae48k_film_measim.png` | 各库 FiLM 增益和偏移曲线 | 4.1.4 Evaluation（FiLM 那段） |
| `ae48k_training.png` | FFM48k_meas 训练曲线（训练 / 验证 RMSE，分库） | 4.1.2，或第 3 章训练那段 |
| `ae48k_figures.json` | 选人结果、关卡、曲线数值、分频带和分仰角汇总、FiLM 和训练曲线的统计 | 写 caption 和正文用 |

**运行**：GPU，从上到下 Run all。数据和检查点都在 Drive 原来的位置。

In [ ]:
# ==========================================================================
# Cell 0 — 路径 · 规则
# ==========================================================================
import os, sys, json, hashlib, zipfile, time, shutil
import numpy as np
from google.colab import drive
drive.mount('/content/drive')
import importlib.util, subprocess
if importlib.util.find_spec('healpy') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'healpy'])

DRIVE = '/content/drive/MyDrive'
CKPT_DIR = f'{DRIVE}/HRTF_Checkpoints'
DATA_ZIP     = f'{DRIVE}/ELEC5021/data_48k_2026-10-10.zip'
ZIP_SHA256   = 'f51c6b072123e70d6d8a0691373ab90e3f6db9856ce6306f65d949c87fbf97c7'
SPLIT_FILE   = f'{DRIVE}/ELEC5021/split_48k.json'
SPLIT_SHA256 = 'dc89bc454fef351951283cccb3772fdd3e552bbdde5d13668f8f576f4cd83561'
LOCAL48 = '/content/data_48k_local'
CKPT_MEAS   = f'{CKPT_DIR}/FFM48k_meas_best.pth'
CKPT_MEASIM = f'{CKPT_DIR}/FFM48k_measim_best.pth'      # 只读 FiLM 参数
OUT_DIR  = f'{CKPT_DIR}/ae48k_figures'
OUT_JSON = f'{OUT_DIR}/ae48k_figures.json'
os.makedirs(OUT_DIR, exist_ok=True)

# 复现关卡：FFM48k_meas 验证集，已记录的 RMSE（dB）—— 与 AE_LSD_Report.ipynb 相同
EXPECT_MODEL = {'pooled': 1.4793, 'CARLab': 1.3581, 'HUTUBS': 1.8178, 'SONICOM': 1.4445, 'SYMARE': 1.3209}
EXPECT_BASE  = {'CARLab': 5.053, 'HUTUBS': 5.163, 'SONICOM': 4.983, 'SYMARE': 4.813}
TOL = 0.005

# 事先定好的选图规则
SUBJECT_RULE = 'median'                              # 每库两耳平均 LSD 的中位数受试者（偶数人取偏小的那个）
FREQS_HZ = (500, 4000, 8000, 12000)                  # HEALPix 图的频率（取最近的频点）
DIRECTIONS = [('Front (az 0°, el 0°)', 0, 0), ('Left (az 90°, el 0°)', 90, 0), ('Right (az −90°, el 0°)', -90, 0),
              ('Left upper (az 45°, el 45°)', 45, 45), ('Right upper (az −45°, el 45°)', -45, 45)]
BANDS = {'0.19-5 kHz': (187.5, 5000.0), '6-12 kHz': (6000.0, 12000.0), '12-16 kHz': (12000.0 + 1e-6, 16000.0)}
DPI = 300
DF, N_BINS = 48000 / 256, 86
FREQS = np.arange(N_BINS) * DF
for p in [DATA_ZIP, SPLIT_FILE, CKPT_MEAS, CKPT_MEASIM]:
    print(f"{'OK ' if os.path.exists(p) else '★缺失'} {p}")


def sha256(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

In [ ]:
!pip install healpy -q

In [ ]:
# ==========================================================================
# Cell 2 - 模型定义（GELU + FiLM）
#
# 与 FFM_2_6_2QuadDB.ipynb（FFM26）的模型定义逐字相同；只有 num_bins（86）和 num_datasets 由 config 决定。
# 架构不要改：FFM48k_*.pth 的权重形状由这里决定。
# ==========================================================================
import numpy as np
import torch
import torch.nn as nn
import healpy as hp


# ---------------------------------------------- spherical convolution
class OSLOConv(nn.Module):
    """Anisotropic convolution on HEALPix: each pixel plus its 8 neighbours, 9 independent weights."""
    def __init__(self, in_channels, out_channels, nside=16):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        neighbors = hp.get_all_neighbours(nside, np.arange(self.npix))
        rf = np.concatenate([neighbors, np.arange(self.npix)[np.newaxis, :]], axis=0)
        rf[rf == -1] = self.npix
        self.register_buffer('neighborhood', torch.tensor(rf, dtype=torch.int64))
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels, 9))
        self.bias = nn.Parameter(torch.empty(out_channels))
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        nn.init.zeros_(self.bias)

    def forward(self, x):
        B, C, _ = x.shape
        pad = torch.zeros(B, C, 1, device=x.device, dtype=x.dtype)
        g = torch.cat([x, pad], dim=2)[:, :, self.neighborhood]
        return torch.einsum('bcqn,ocq->bon', g, self.weight) + self.bias.view(1, -1, 1)


class OSLOResBlock(nn.Module):
    """Frequency-isolated residual block, operating on (B*F, C, N).
    * v2: ReLU -> GELU. HRTF log-magnitude is bipolar (negative means attenuation), so
      ReLU truncates the negative half; GELU is smooth, keeps negatives, and matches the fc layers."""
    def __init__(self, in_channels, out_channels, nside=16, num_groups=8):
        super().__init__()
        self.shortcut = (nn.Conv1d(in_channels, out_channels, 1)
                         if in_channels != out_channels else nn.Identity())
        self.conv1 = OSLOConv(in_channels, out_channels, nside)
        self.gn1 = nn.GroupNorm(num_groups, out_channels)
        self.conv2 = OSLOConv(out_channels, out_channels, nside)
        self.gn2 = nn.GroupNorm(num_groups, out_channels)
        self.act = nn.GELU()

    def forward(self, x):
        res = self.shortcut(x)
        out = self.act(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        return self.act(out + res)


# ---------------------------------------------- per-dataset FiLM
class DatasetFiLM(nn.Module):
    """Learn one PER-FREQUENCY scale/shift curve for each dataset.

    ** Constraint 1: frequency only, NEVER direction.
      A per-frequency affine map is one EQ curve (93 dof per dataset). All it can absorb is
      "colouration common to the whole dataset and independent of direction" — loudspeaker and
      microphone response, equalisation convention, overall level. By construction it CANNOT
      express a direction-dependent difference, so population anatomy (ear-size distributions
      differ across databases) must still live in z. Contrast: subtracting a per-database mean
      over (frequency x direction) is 3072x93 dof, enough to remove the real population structure
      as well — exactly the wrong direction Craig pointed at. A factor of 3072 in expressive power.

    ** Constraint 2: centre across datasets.
      Without centring, log_gamma/beta learn a global EQ that cancels against the network's own
      output gain — a redundant degree of freedom. After the cross-database mean is removed FiLM
      expresses only RELATIVE differences; the absolute part stays in the network. Exactly the identity when num_datasets=1.
    """
    def __init__(self, num_datasets, num_bins):
        super().__init__()
        self.num_datasets = num_datasets
        self.log_gamma = nn.Parameter(torch.zeros(num_datasets, num_bins))
        self.beta = nn.Parameter(torch.zeros(num_datasets, num_bins))

    def params(self, dataset_id):
        lg = self.log_gamma - self.log_gamma.mean(0, keepdim=True)
        bt = self.beta - self.beta.mean(0, keepdim=True)
        return torch.exp(lg[dataset_id]).unsqueeze(-1), bt[dataset_id].unsqueeze(-1)

    def forward(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return x * g + b

    def inverse(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return (x - b) / g


# ---------------------------------------------- encoder / decoder

class Isolated_OSLO_Encoder(nn.Module):
    """Encode each frequency bin independently -> latent (B, F, latent_dim)"""
    def __init__(self, nside=16, latent_dim_per_freq=64):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.enc1 = OSLOResBlock(1, 16, nside)
        self.enc2 = OSLOResBlock(16, 32, nside)
        self.fc_encode = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * self.npix, 512), nn.GELU(),
            nn.Linear(512, self.latent_dim))

    def forward(self, x):
        B, F, N = x.shape
        z = self.fc_encode(self.enc2(self.enc1(x.reshape(B * F, 1, N))))
        return z.reshape(B, F, self.latent_dim)


class Conditioned_OSLO_Decoder(nn.Module):
    """FFM frequency-conditioned decoder + per-dataset FiLM on the output"""
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.freq_embed_dim = freq_embed_dim
        self.ffm_B = nn.Parameter(torch.randn(freq_embed_dim // 2))
        self.fc_decode = nn.Sequential(
            nn.Linear(self.latent_dim + freq_embed_dim, 512), nn.GELU(),
            nn.Linear(512, 32 * self.npix))
        self.dec1 = OSLOResBlock(32, 16, nside)
        self.dec_final = OSLOConv(16, 1, nside)
        self.film = DatasetFiLM(num_datasets, num_bins)

    def get_freq_embedding(self, num_freqs, device):
        f = torch.arange(num_freqs, device=device).float() / (num_freqs - 1)
        proj = 2 * np.pi * f.unsqueeze(1) * self.ffm_B.unsqueeze(0)
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=1)

    def forward(self, z, dataset_id=None):
        B, F, _ = z.shape
        fe = self.get_freq_embedding(F, z.device).unsqueeze(0).expand(B, -1, -1)
        h = torch.cat([z, fe], dim=-1).reshape(B * F, -1)
        h = self.fc_decode(h).reshape(B * F, 32, self.npix)
        out = self.dec_final(self.dec1(h)).reshape(B, F, self.npix)
        if dataset_id is not None:
            out = self.film(out, dataset_id)
        return out


class FFM_OSLO_Autoencoder(nn.Module):
    """One SHARED-parameter inverse FiLM at the encoder input — zero extra parameters.

    With FiLM on the decoder output only, the colouration stays in x, the encoder takes it in, and it
    lands in z — but z is exactly what the downstream geometry->latent stage needs. Output-side FiLM
    only makes the reconstruction loss line up; it does not clean z.

    Here     x_norm = (x - beta)/gamma        <- remove colouration
             z      = encoder(x_norm)
             out    = gamma * decoder(z) + beta   <- back to the original space, loss still on raw

    The SAME FiLM parameters are used. Together with centring this also kills the trivial degenerate
    direction "scale gamma up, scale the network down". Set tie_encoder_film=False to disable.
    """
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93, tie_encoder_film=True):
        super().__init__()
        self.encoder = Isolated_OSLO_Encoder(nside, latent_dim_per_freq)
        self.decoder = Conditioned_OSLO_Decoder(nside, latent_dim_per_freq,
                                                freq_embed_dim, num_datasets, num_bins)
        self.tie_encoder_film = tie_encoder_film

    def forward(self, x, dataset_id=None):
        if dataset_id is not None and self.tie_encoder_film:
            x = self.decoder.film.inverse(x, dataset_id)
        z = self.encoder(x)
        return self.decoder(z, dataset_id), z


print('model defined (FFM + FiLM)')

In [ ]:
# ==========================================================================
# Cell 3 — 数据（只建 train / val，不建 test）· 载入 FFM48k_meas · 读 FiLM 和训练曲线
# ==========================================================================
from torch.utils.data import Dataset
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cuda.matmul.allow_tf32 = True          # 与训练 / 评估 notebook 相同，才能复现记录的 RMSE
torch.backends.cudnn.allow_tf32 = True
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

t0 = time.time()
ROOT = f'{LOCAL48}/data_48k'; STAMP = f'{LOCAL48}/.verified_{ZIP_SHA256[:16]}'
if not os.path.exists(STAMP):
    assert sha256(DATA_ZIP) == ZIP_SHA256, 'data_48k zip 的 SHA-256 不符'
    if os.path.exists(LOCAL48): shutil.rmtree(LOCAL48)
    os.makedirs(LOCAL48)
    with zipfile.ZipFile(DATA_ZIP) as z: z.extractall(LOCAL48)
    n = 0
    for name in sorted(os.listdir(ROOT)):
        if not name.endswith('_48k'): continue
        for m in json.load(open(f'{ROOT}/{name}/manifest.json')):
            for side in ('left', 'right'):
                f = f'{ROOT}/{name}/npy_{side}/{m[side]}'
                assert sha256(f) == m[f'{side}_sha256'], f'SHA-256 不符: {f}'; n += 1
    open(STAMP, 'w').write(ZIP_SHA256)
    print(f'zip 和 {n} 个数组的 SHA-256 核对通过（{time.time() - t0:.0f}s）')
assert sha256(SPLIT_FILE) == SPLIT_SHA256, 'split_48k.json 的 SHA-256 不符'
SRC = json.load(open(SPLIT_FILE))
MIRROR = np.load(f'{ROOT}/healpix_mirror_idx.npy')
assert np.array_equal(MIRROR[MIRROR], np.arange(3072)), '镜像不是对合'

SPEC = [dict(id=0, tag='CARLab'), dict(id=1, tag='HUTUBS'), dict(id=2, tag='SONICOM'), dict(id=3, tag='SYMARE')]
D48, SPLIT48 = [], {}
for s in SPEC:
    d = f"{ROOT}/{s['tag']}_48k"; ids = sorted(m['id'] for m in json.load(open(f'{d}/manifest.json')))
    where = {sid: k for k in ('train', 'val', 'test') for sid in SRC[s['tag']][k]}
    SPLIT48[s['tag']] = {k: sorted(i for i in ids if where[i] == k) for k in ('train', 'val')}   # ★ 不取 test
    D48.append(dict(s, left=f'{d}/npy_left', right=f'{d}/npy_right'))
EXPECTED = {'CARLab': (174, 22), 'HUTUBS': (58, 14), 'SONICOM': (298, 40), 'SYMARE': (37, 9)}
for t, (ntr, nva) in EXPECTED.items():
    assert (len(SPLIT48[t]['train']), len(SPLIT48[t]['val'])) == (ntr, nva), t
assert all(set(v) == {'train', 'val'} for v in SPLIT48.values()), '不应出现 test 分区'
print('划分人数与 FFM48k_Train 相同（train / val）；测试分区未创建')


class Ear48(Dataset):
    # 与 AE_LSD_Report 的 Ear48 相同，另外返回 (受试者, 哪只耳)
    def __init__(self, d, ids, mirror=MIRROR, n_bins=N_BINS):
        self.items, self.mirror, self.n_bins = [], mirror, n_bins
        for sid in sorted(ids):
            for side, sfx in (('left', '_left_full.npy'), ('right', '_right_full.npy')):
                p = os.path.join(d[side], f'{sid}{sfx}'); assert os.path.exists(p), p
                self.items.append((sid, side, p))
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        sid, side, p = self.items[i]
        x = np.load(p).astype(np.float32)[:self.n_bins]
        if side == 'right': x = x[:, self.mirror]
        return torch.from_numpy(x), sid, side


ck = torch.load(CKPT_MEAS, map_location=device, weights_only=False)
cfg = ck['config']
assert cfg['data_zip_sha256'] == ZIP_SHA256 and cfg['split_sha256'] == SPLIT_SHA256 and ck.get('run_name') == 'FFM48k_meas'
assert cfg['num_bins'] == N_BINS and cfg['num_datasets'] == 4
for t in SPLIT48:
    assert all(ck['split'][t][k] == SPLIT48[t][k] for k in ('train', 'val')), f'{t} 的划分与检查点不同'
model = FFM_OSLO_Autoencoder(nside=cfg['nside'], latent_dim_per_freq=cfg['latent_dim'], freq_embed_dim=cfg['freq_embed_dim'],
                             num_datasets=cfg['num_datasets'], num_bins=cfg['num_bins'], tie_encoder_film=cfg['tie_encoder_film']).to(device)
model.load_state_dict(ck['model_state_dict']); model.eval()
HISTORY, CK_EPOCH, CK_VAL = ck.get('history', {}), ck['epoch'], ck['val_rmse']
TRAIN_CFG = {k: cfg.get(k) for k in ('lr', 'batch_size', 'weight_decay', 'w_mse', 'w_spectral', 'amp', 'latent_dim', 'freq_embed_dim', 'nside')}
print(f"载入 FFM48k_meas_best.pth | epoch {CK_EPOCH + 1} | 记录的验证 RMSE {CK_VAL:.4f} dB | 训练曲线 {len(HISTORY.get('val_rmse', []))} 轮")
del ck


def film_curves(state, n_db):
    # 与模型里 DatasetFiLM.params() 相同：跨库中心化之后的 log γ 和 β；返回 (20 log10 γ [dB], β [dB])
    lg = state['decoder.film.log_gamma'].double().cpu().numpy()[:n_db]
    bt = state['decoder.film.beta'].double().cpu().numpy()[:n_db]
    lg = lg - lg.mean(0, keepdims=True); bt = bt - bt.mean(0, keepdims=True)
    return 20 * np.log10(np.exp(lg)), bt


FILM = {'FFM48k_meas': dict(tags=[d['tag'] for d in D48], curves=film_curves(model.state_dict(), 4))}
ck2 = torch.load(CKPT_MEASIM, map_location='cpu', weights_only=False)
assert ck2.get('run_name') == 'FFM48k_measim' and ck2['config']['num_datasets'] == 6
FILM['FFM48k_measim'] = dict(tags=['CARLab', 'HUTUBS', 'SONICOM', 'SYMARE', 'SONICOM_SIM', 'SYMARE_BEM'],
                             curves=film_curves(ck2['model_state_dict'], 6))
del ck2
print('FiLM 参数已读（FFM48k_meas 4 行，FFM48k_measim 6 行）')

In [ ]:
# ==========================================================================
# Cell 4 — 一次遍历验证集：逐频点、逐方向、逐耳的误差（模型与训练集均值基线同一批耳朵）· 复现关卡
# ==========================================================================
_th, _ = hp.pix2ang(16, np.arange(3072))
ELEV = 90 - np.degrees(_th)
MEAS = ELEV >= -45.0
assert MEAS.sum() == 2652


@torch.no_grad()
def train_mean(part, bs=8):
    acc, n = None, 0
    for b in range(0, len(part), bs):
        x = torch.stack([part[i][0] for i in range(b, min(b + bs, len(part)))]).to(device).double()
        acc = x.sum(0) if acc is None else acc + x.sum(0); n += len(x)
    return (acc / n).float()


STATS, MU = {}, {}
with torch.no_grad():
    for d in D48:
        tag, did = d['tag'], d['id']
        MU[tag] = train_mean(Ear48(d, SPLIT48[tag]['train']))
        part = Ear48(d, SPLIT48[tag]['val'])
        st = dict(se_bin_m=np.zeros(N_BINS), se_bin_b=np.zeros(N_BINS), se_dir_m=np.zeros(3072), se_dir_b=np.zeros(3072),
                  ears=0, ear_lsd={})
        for b in range(0, len(part), 8):
            items = [part[i] for i in range(b, min(b + 8, len(part)))]
            x = torch.stack([it[0] for it in items]).to(device)
            rec, _ = model(x, torch.full((len(x),), did, dtype=torch.long, device=device))
            e2m = ((rec.float() - x).double() ** 2); e2b = ((MU[tag][None] - x).double() ** 2)
            st['se_bin_m'] += e2m.sum((0, 2)).cpu().numpy(); st['se_bin_b'] += e2b.sum((0, 2)).cpu().numpy()
            st['se_dir_m'] += e2m.sum((0, 1)).cpu().numpy(); st['se_dir_b'] += e2b.sum((0, 1)).cpu().numpy()
            lsd_m = torch.sqrt(e2m.mean(1)).mean(1).cpu().numpy(); lsd_b = torch.sqrt(e2b.mean(1)).mean(1).cpu().numpy()
            for (xx, sid, side), lm, lb in zip(items, lsd_m, lsd_b):
                st['ear_lsd'][(sid, side)] = (float(lm), float(lb))
            st['ears'] += len(x)
        STATS[tag] = st
        print(f"{tag:<9} {st['ears']:>3} 只耳 | 模型 RMSE {np.sqrt(st['se_bin_m'].sum() / (st['ears'] * N_BINS * 3072)):.4f}"
              f" | 基线 {np.sqrt(st['se_bin_b'].sum() / (st['ears'] * N_BINS * 3072)):.4f} dB")

rmse = lambda se, ears, n: float(np.sqrt(se / (ears * n)))
GOT_M = {t: rmse(s['se_bin_m'].sum(), s['ears'], N_BINS * 3072) for t, s in STATS.items()}
GOT_M['pooled'] = rmse(sum(s['se_bin_m'].sum() for s in STATS.values()), sum(s['ears'] for s in STATS.values()), N_BINS * 3072)
GOT_B = {t: rmse(s['se_bin_b'].sum(), s['ears'], N_BINS * 3072) for t, s in STATS.items()}
bad = {k: (round(GOT_M[k], 4), v) for k, v in EXPECT_MODEL.items() if abs(GOT_M[k] - v) > TOL}
bad.update({f'{k} base': (round(GOT_B[k], 4), v) for k, v in EXPECT_BASE.items() if abs(GOT_B[k] - v) > TOL})
assert not bad, f'复现不了已记录的验证集 RMSE {bad} —— 数据、划分或检查点不对，停止'
print(f'✓ 复现关卡通过：{len(EXPECT_MODEL) + len(EXPECT_BASE)} 个已记录的 RMSE 全部在 {TOL} dB 以内')

# ---- 每库中位数受试者（两耳平均 LSD），选之前不看图
PICK = {}
for tag, st in STATS.items():
    subj = sorted({sid for sid, _ in st['ear_lsd']})
    per = sorted(((np.mean([st['ear_lsd'][(s, e)][0] for e in ('left', 'right')]), s) for s in subj))
    lsd, sid = per[(len(per) - 1) // 2]
    PICK[tag] = dict(subject=sid, lsd=float(lsd), rank=(len(per) - 1) // 2 + 1, n=len(per),
                     all=[(s, round(float(v), 4)) for v, s in per])
    print(f'{tag:<9} 中位数受试者 {sid}（{PICK[tag]["rank"]}/{len(per)}），两耳平均 LSD {lsd:.3f} dB')

In [ ]:
# ==========================================================================
# Cell 5 — 画图函数（只用 numpy / matplotlib / healpy；输入都是 dB 数组）
# ==========================================================================
import numpy as np
import healpy as hp
import matplotlib
import matplotlib.pyplot as plt

plt.rcParams.update({'font.family': 'serif', 'mathtext.fontset': 'stix', 'font.size': 9, 'axes.titlesize': 9.5,
                     'axes.labelsize': 9, 'legend.fontsize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8,
                     'savefig.facecolor': 'white'})
COL = {'CARLab': '#1f77b4', 'HUTUBS': '#d62728', 'SONICOM': '#2ca02c', 'SYMARE': '#9467bd',
       'SONICOM_SIM': '#2ca02c', 'SYMARE_BEM': '#9467bd'}
NAME = {'CARLab': 'CAR Lab', 'HUTUBS': 'HUTUBS', 'SONICOM': 'SONICOM', 'SYMARE': 'SYMARE',
        'SONICOM_SIM': 'SONICOM simulations', 'SYMARE_BEM': 'SYMARE BEM'}
NOTCH = (6.0, 12.0)


def save(fig, path):
    fig.savefig(path + '.png', dpi=DPI, bbox_inches='tight')
    fig.savefig(path + '.pdf', bbox_inches='tight')
    plt.close(fig)
    print('  ->', os.path.basename(path) + '.png / .pdf')


def pix_of(az_deg, el_deg, nside=16):
    return int(hp.ang2pix(nside, np.radians(90.0 - el_deg), np.radians(az_deg % 360.0)))


def plot_directions(gt_l, rec_l, gt_r, rec_r, freqs, directions, title, path):
    # 5 个方向竖排一列：两耳实测（实线）与重建（虚线），各自在本人的原始坐标系里
    fk = freqs / 1000
    fig, axes = plt.subplots(len(directions), 1, figsize=(6.3, 1.7 * len(directions) + 0.7), sharex=True, layout='constrained')
    for ax, (name, az, el) in zip(axes, directions):
        p = pix_of(az, el)
        ax.axvspan(*NOTCH, color='0.92', zorder=0)
        ax.plot(fk, gt_l[:, p], color='#1f77b4', lw=1.4, label='left ear, measured')
        ax.plot(fk, rec_l[:, p], color='#5fc3ff', lw=1.3, ls='--', label='left ear, reconstructed')
        ax.plot(fk, gt_r[:, p], color='#d62728', lw=1.4, label='right ear, measured')
        ax.plot(fk, rec_r[:, p], color='#ff9a3c', lw=1.3, ls='--', label='right ear, reconstructed')
        ax.set_title(name, loc='left'); ax.set_ylabel('dB'); ax.grid(alpha=.3, lw=.5)
        ax.set_xlim(0, fk[-1])
    axes[-1].set_xlabel('frequency (kHz)')
    fig.suptitle(title, fontsize=10)
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc='outside lower center', ncol=2, frameon=False)
    save(fig, path)


def _el45_line(color):
    phi = np.linspace(-np.pi, np.pi, 361)
    hp.projplot(np.full_like(phi, np.radians(135.0)), phi, color=color, ls='--', lw=0.8)


def _labels(color):
    for lon, txt in ((0, 'F'), (90, 'L'), (-90, 'R')):
        hp.projtext(lon, 0, txt, lonlat=True, color=color, fontsize=7, ha='center', va='center')


def plot_healpix(gt, rec, freqs, targets_hz, title, path):
    # 每行一个频率：实测 / 重建 / |误差|。实测和重建同一色标；左右视图中心是正前方，左边是听者左侧
    rows = len(targets_hz)
    fig = plt.figure(figsize=(7.2, 1.75 * rows + 0.4))
    picked = []
    for r, hz in enumerate(targets_hz):
        k = int(np.argmin(np.abs(freqs - hz))); f = freqs[k]
        g, p = gt[k], rec[k]; e = np.abs(g - p)
        lo, hi = float(min(g.min(), p.min())), float(max(g.max(), p.max()))
        picked.append(dict(target_hz=hz, bin=k, freq_hz=float(f), mae=float(e.mean()), max_err=float(e.max())))
        for c, (m, ttl, cmap, vmin, vmax, lc) in enumerate((
                (g, 'measured (dB)', 'viridis', lo, hi, 'white'), (p, 'reconstructed (dB)', 'viridis', lo, hi, 'white'),
                (e, f'|error|, mean {e.mean():.2f} dB', 'hot', 0.0, float(e.max()), 'cyan'))):
            hp.mollview(m, fig=fig.number, sub=(rows, 3, 3 * r + c + 1), title=f'{f / 1000:.2f} kHz, {ttl}',
                        cmap=cmap, min=vmin, max=vmax, unit='', format='%.0f', cbar=True, notext=False)
            _el45_line(lc)
    fig.suptitle(title, fontsize=10, y=1.03)
    save(fig, path)
    return picked


def plot_error_by_frequency(curves, freqs, path):
    # curves[tag] = (model RMSE per bin, baseline RMSE per bin)
    fk = freqs / 1000
    fig, ax = plt.subplots(1, 2, figsize=(7.2, 2.9), layout='constrained')
    for tag, (m, b) in curves.items():
        ax[0].plot(fk, m, color=COL[tag], lw=1.4, label=NAME[tag])
        ax[0].plot(fk, b, color=COL[tag], lw=1.0, ls='--', alpha=.8)
        ax[1].plot(fk[1:], 100 * m[1:] / b[1:], color=COL[tag], lw=1.4, label=NAME[tag])
    for a in ax:
        a.axvspan(*NOTCH, color='0.92', zorder=0); a.set_xlabel('frequency (kHz)'); a.grid(alpha=.3, lw=.5)
        a.set_xlim(0, fk[-1])
    ax[0].set_ylabel('RMSE (dB)'); ax[0].set_title('reconstruction (solid) and mean HRTF (dashed)', loc='left')
    ax[1].set_ylabel('reconstruction / mean (%)'); ax[1].set_title('error relative to the mean HRTF', loc='left')
    ax[1].set_ylim(bottom=0)
    h, l = ax[0].get_legend_handles_labels()
    fig.legend(h, l, loc='outside lower center', ncol=len(curves), frameon=False)
    save(fig, path)


def plot_error_maps(maps, path):
    # maps[tag] = 逐方向 RMSE（3072）；所有耳都在左耳坐标系（右耳已镜像），所以图的左半边是同侧
    vmax = float(max(m.max() for m in maps.values()))
    n = len(maps)
    fig = plt.figure(figsize=(7.2, 2.4 * ((n + 1) // 2) + 0.3))
    for i, (tag, m) in enumerate(maps.items()):
        hp.mollview(m, fig=fig.number, sub=((n + 1) // 2, 2, i + 1), title=f'{NAME[tag]}: RMSE (dB), mean {m.mean():.2f}',
                    cmap='magma', min=0.0, max=vmax, unit='', format='%.1f', cbar=True)
        _el45_line('cyan')
    save(fig, path)


def plot_film(gain_db, beta, tags, freqs, title, path):
    # DC 频点不画（β 的峰值常常落在 DC）；仿真用虚线
    fk = freqs[1:] / 1000
    fig, ax = plt.subplots(1, 2, figsize=(7.2, 2.9), layout='constrained')
    for k, tag in enumerate(tags):
        ls = '--' if tag.endswith(('_SIM', '_BEM')) else '-'
        ax[0].plot(fk, gain_db[k, 1:], color=COL[tag], ls=ls, lw=1.4, label=NAME[tag])
        ax[1].plot(fk, beta[k, 1:], color=COL[tag], ls=ls, lw=1.4, label=NAME[tag])
    ax[0].set_title(r'gain $20\log_{10}\gamma$ (dB)', loc='left'); ax[1].set_title(r'offset $\beta$ (dB)', loc='left')
    for a in ax:
        a.axhline(0, color='0.5', lw=.6); a.set_xlabel('frequency (kHz)'); a.grid(alpha=.3, lw=.5); a.set_xlim(0, fk[-1])
    fig.suptitle(title, fontsize=10)
    h, l = ax[0].get_legend_handles_labels()
    fig.legend(h, l, loc='outside lower center', ncol=min(len(tags), 4), frameon=False)
    save(fig, path)


def plot_training(hist, tags, path):
    ep = np.arange(1, len(hist['val_rmse']) + 1)
    best = int(np.argmin(hist['val_rmse']))
    fig, ax = plt.subplots(figsize=(6.3, 3.2), layout='constrained')
    for t in tags:
        ax.plot(ep, [h[t] for h in hist['per_dataset']], color=COL[t], lw=.9, alpha=.8, label=f'validation, {NAME[t]}')
    ax.plot(ep, hist['train_rmse'], color='k', lw=1.5, ls='--', label='training, pooled')
    ax.plot(ep, hist['val_rmse'], color='k', lw=1.8, label='validation, pooled')
    ax.axvline(best + 1, color='0.4', ls=':', lw=1)
    k = min(4, len(ep) - 1)
    top = max([hist['val_rmse'][k], hist['train_rmse'][k]] + [hist['per_dataset'][k][t] for t in tags]) + 0.1
    ax.set_ylim(min(hist['train_rmse']) - 0.1, top)
    ax.set_xlabel('epoch'); ax.set_ylabel('RMSE (dB)'); ax.grid(alpha=.3, lw=.5)
    h, l = ax.get_legend_handles_labels()
    fig.legend(h, l, loc='outside lower center', ncol=3, frameon=False)
    save(fig, path)
    return best

In [ ]:
# ==========================================================================
# Cell 6 — 出图（数字已在 Cell 4 记录并通过关卡）
# ==========================================================================
OUT = {}


@torch.no_grad()
def reconstruct_subject(d, sid):
    # 两只耳都回到本人原始坐标系：右耳镜像进模型，输出再镜像回来（镜像是对合）
    res = {}
    for side, sfx in (('left', '_left_full.npy'), ('right', '_right_full.npy')):
        x = np.load(os.path.join(d[side], f'{sid}{sfx}')).astype(np.float32)[:N_BINS]
        xin = x[:, MIRROR] if side == 'right' else x
        rec, _ = model(torch.from_numpy(xin)[None].to(device), torch.full((1,), d['id'], dtype=torch.long, device=device))
        rec = rec[0].float().cpu().numpy()
        res[side] = (x, rec[:, MIRROR] if side == 'right' else rec)
    return res


for d in D48:
    tag, sid = d['tag'], PICK[d['tag']]['subject']
    r = reconstruct_subject(d, sid)
    (gl, rl), (gr, rr) = r['left'], r['right']
    # 关卡：画图用的重建（两耳各回到自己的坐标系）必须给出 Cell 4 选人时记录的 LSD —— 镜像写错会在这里被拦下
    lsd_now = np.mean([np.sqrt(((g - p) ** 2).mean(0)).mean() for g, p in ((gl, rl), (gr, rr))])
    assert abs(lsd_now - PICK[tag]['lsd']) < 1e-3, f'{tag} {sid}: 画图用的重建 LSD {lsd_now:.4f} ≠ 记录的 {PICK[tag]["lsd"]:.4f}'
    PICK[tag]['lsd_check'] = float(lsd_now)
    disp = sid.split('_', 1)[-1]
    ttl = f"{NAME[tag]} {disp}: validation subject with the median LSD ({PICK[tag]['lsd']:.2f} dB)"
    plot_directions(gl, rl, gr, rr, FREQS, DIRECTIONS, ttl, f'{OUT_DIR}/ae48k_directions_{tag}')
    PICK[tag]['healpix'] = plot_healpix(gl, rl, FREQS, FREQS_HZ, f'{NAME[tag]} {disp}, left ear', f'{OUT_DIR}/ae48k_healpix_{tag}')
    PICK[tag]['direction_rmse'] = {name: dict(left=float(np.sqrt(((gl - rl)[:, pix_of(az, el)] ** 2).mean())),
                                              right=float(np.sqrt(((gr - rr)[:, pix_of(az, el)] ** 2).mean())))
                                   for name, az, el in DIRECTIONS}

CURVES = {t: (np.sqrt(s['se_bin_m'] / (s['ears'] * 3072)), np.sqrt(s['se_bin_b'] / (s['ears'] * 3072))) for t, s in STATS.items()}
plot_error_by_frequency(CURVES, FREQS, f'{OUT_DIR}/ae48k_error_by_frequency')
MAPS = {t: np.sqrt(s['se_dir_m'] / (s['ears'] * N_BINS)) for t, s in STATS.items()}
plot_error_maps(MAPS, f'{OUT_DIR}/ae48k_error_maps')
for run, f in FILM.items():
    plot_film(*f['curves'], f['tags'], FREQS, f'{run}: per-database FiLM', f'{OUT_DIR}/ae48k_film_{run.split("_")[1]}')
BEST = plot_training(HISTORY, [d['tag'] for d in D48], f'{OUT_DIR}/ae48k_training') if HISTORY.get('val_rmse') else None

In [ ]:
# ==========================================================================
# Cell 7 — 导出 JSON（写 caption 和正文用）· 打包下载
# ==========================================================================
def band_rmse(se_bin, ears, lo, hi):
    m = (FREQS >= lo) & (FREQS <= hi)
    return float(np.sqrt(se_bin[m].sum() / (ears * 3072 * m.sum())))


def el_rmse(se_dir, ears, mask):
    return float(np.sqrt(se_dir[mask].sum() / (ears * N_BINS * mask.sum())))


summary = {}
for t, s in STATS.items():
    summary[t] = dict(
        ears=s['ears'], rmse_model=GOT_M[t], rmse_baseline=GOT_B[t],
        bands={b: dict(model=band_rmse(s['se_bin_m'], s['ears'], lo, hi), baseline=band_rmse(s['se_bin_b'], s['ears'], lo, hi))
               for b, (lo, hi) in BANDS.items()},
        elevation={'el>=-45': dict(model=el_rmse(s['se_dir_m'], s['ears'], MEAS), baseline=el_rmse(s['se_dir_b'], s['ears'], MEAS)),
                   'el<-45': dict(model=el_rmse(s['se_dir_m'], s['ears'], ~MEAS), baseline=el_rmse(s['se_dir_b'], s['ears'], ~MEAS))},
        curve_model=[round(float(v), 4) for v in CURVES[t][0]], curve_baseline=[round(float(v), 4) for v in CURVES[t][1]],
        map_quantiles={q: round(float(np.quantile(MAPS[t], q / 100)), 4) for q in (5, 50, 95)})

film_stats = {}
for run, f in FILM.items():
    g, b = f['curves']
    film_stats[run] = {tag: dict(peak_abs_gain_db=float(np.abs(g[k]).max()), peak_abs_gain_db_no_dc=float(np.abs(g[k, 1:]).max()),
                                 beta_dc=float(b[k, 0]), beta_mean_ge500hz=float(b[k, 3:].mean()),
                                 beta_second_diff_rms_no_dc=float(np.sqrt(np.mean(np.diff(b[k, 1:], 2) ** 2))),
                                 beta_second_diff_rms_ge500hz=float(np.sqrt(np.mean(np.diff(b[k, 3:], 2) ** 2))))
                       for k, tag in enumerate(f['tags'])}

train_stats = None
if BEST is not None:
    h = HISTORY
    train_stats = dict(epochs=len(h['val_rmse']), best_epoch=BEST + 1, checkpoint_epoch=CK_EPOCH + 1,
                       val_at_best=float(h['val_rmse'][BEST]), train_at_best=float(h['train_rmse'][BEST]),
                       gap=float(h['val_rmse'][BEST] - h['train_rmse'][BEST]),
                       val_last10=[round(float(v), 4) for v in h['val_rmse'][-10:]],
                       per_dataset_at_best={t: float(v) for t, v in h['per_dataset'][BEST].items()},
                       settings=TRAIN_CFG)

OUTJ = dict(
    created=time.strftime('%Y-%m-%d %H:%M:%S'), notebook='notebooks/FFM48k_Figures.ipynb', model='FFM48k_meas_best.pth',
    partition='validation only; the FFM48k_meas test partition was not created',
    gate=dict(model={k: round(v, 4) for k, v in GOT_M.items()}, baseline={k: round(v, 4) for k, v in GOT_B.items()}, tol=TOL),
    rules=dict(subject=SUBJECT_RULE + ' two-ear LSD per database (lower middle if even)', freqs_hz=FREQS_HZ,
               directions=[(n, az, el) for n, az, el in DIRECTIONS], bands=BANDS,
               frames='subject figures in each ear\'s own frame; error maps in the left-ear frame (right ears mirrored)'),
    bins=dict(n=N_BINS, df_hz=DF, top_hz=float(FREQS[-1])),
    picked={t: {k: v for k, v in p.items()} for t, p in PICK.items()},
    summary=summary, film=film_stats, training=train_stats,
    figures=sorted(f for f in os.listdir(OUT_DIR) if f.endswith(('.png', '.pdf'))))
json.dump(OUTJ, open(OUT_JSON, 'w'), indent=1, default=float)
print(f'-> {OUT_JSON}')

for t, s in summary.items():
    print(f"{t:<9} RMSE {s['rmse_model']:.3f}/{s['rmse_baseline']:.3f} | "
          + ' | '.join(f"{b} {v['model']:.2f}/{v['baseline']:.2f}" for b, v in s['bands'].items())
          + f" | el≥−45 {s['elevation']['el>=-45']['model']:.2f}, el<−45 {s['elevation']['el<-45']['model']:.2f}")
if train_stats:
    print(f"best epoch {train_stats['best_epoch']} | train {train_stats['train_at_best']:.4f} | val {train_stats['val_at_best']:.4f}"
          f" | gap {train_stats['gap']:+.4f} dB")

zpath = '/content/ae48k_figures.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUT_DIR)):
        z.write(f'{OUT_DIR}/{f}', f)
print(f'-> {zpath}（{os.path.getsize(zpath) / 1e6:.1f} MB）')
from google.colab import files
files.download(zpath)

## 跑完之后

把 `ae48k_figures.zip` 解压到 `hrtf-GitHub/results/ae48k_figures/`（JSON 也在里面），告诉我一声。我会：
核对 JSON 里的数（关卡、选到的人、分频带和分仰角汇总、FiLM 和训练曲线统计），
给每张图写 caption 草稿和放置位置，并把 PROJECT_LOG 补一条。论文要用的 PNG 再拷到 Overleaf 的 `figures/`。